In [ ]:
import numpy as np

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit import Qubit, Clbit, marginal_counts
from qiskit.circuit.library import HGate, XGate, SGate, SdgGate
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit.result import marginal_counts
from qiskit.visualization import plot_histogram

In [57]:
# Magic numbers
OPT_LEVEL = 0
SHOTS = 4096
SEED = 7
LAM1, LAM2 = 0.005, 0.02
P = 0.03

# default collections
NOISE = ("ideal", "gates", "readout", "combined")
BASIS_1Q = ['h','x','z','s','sdg','ry','rz','id']
BASIS_2Q = ['cx']
BASIS = BASIS_1Q + BASIS_2Q

In [ ]:
# Helper functions
def add_noise(kind: str):
    # helper to add noise to the AerSimulator
    nm = NoiseModel()
    if kind in ("gates", "combined"):
        nm.add_all_qubit_quantum_error(depolarizing_error(LAM1, 1), BASIS_1Q)
        nm.add_all_qubit_quantum_error(depolarizing_error(LAM2, 2), BASIS_2Q)
    if kind in ("readout", "combined"):
        nm.add_all_qubit_readout_error(ReadoutError([[1 - P, P],[P, 1 - P]]))
    return nm

def run(qc: QuantumCircuit, 
        noise: str, 
        shots: int = SHOTS, 
        seed: int = SEED, 
        opt_level: int = OPT_LEVEL):
    # helper to run the thing
    sim = AerSimulator(method="density_matrix",
                       noise_model= add_noise(noise),
                       basis_gates=BASIS)
    tqc = transpile(qc, sim, optimization_level=opt_level, seed_transpiler=seed)
    result = sim.run(tqc, shots=shots, seed_simulator=seed).result()
    return result, tqc

# Problem A: Quantum Teleportation

In [41]:
PROBES = {
    "0": [],
    "1": [XGate()],
    "+": [HGate()],
    "-": [XGate(), HGate()],
    "+i": [HGate(), SGate()],
    "-i": [HGate(), SdgGate()]
}

# get the preparation and inverse-preparation stuff
def get_prep_circuits(probe):
    prep = QuantumCircuit(1, name=f"prep_probe")
    for gate in PROBES[probe]:
        prep.append(gate, [0])
    post = prep.inverse()
    post.name = f"post_probe"
    return prep.to_gate(), post.to_gate()

def quantum_teleportation(probe, bob_corrects=True):
    # q[0] = qubit to be sent, q[1..2] = e-bits, q[3..5] = clbit
    # bob_corrects is whether bob will correct his qubit
    q = QuantumCircuit(3, 3)

    # prepare qubit
    prep, post = get_prep_circuits(probe)
    q.append(prep, [0])
    q.barrier()

    # create entangled pair between q[1] and q[2]
    q.h(1)
    q.cx(1, 2)

    # entangle "unknown" probe state
    q.cx(0, 1)

    # measure on alice side
    q.h(0)
    q.measure(0, 0)
    q.measure(1, 1)

    # bob corrects
    if bob_corrects:
        with q.if_test((q.clbits[0], 1)):
            q.z(2)
        with q.if_test((q.clbits[1], 1)):
            q.x(2)

    # post-processing + measure
    q.barrier()
    q.append(post, [2])
    q.measure(2, 2)
    return q

## Circuit
Using an arbitrary 0 starting state and assuming Bob corrects as usual:
- `prep_probe` prepares the probe
- `post_probe` post-processes Bob's qubit (with the inverse preparation operation)

In [55]:
quantum_teleportation("0").draw()

┌────────────┐ ░                ┌───┐┌─┐                          »
q_0: ┤ prep_probe ├─░─────────────■──┤ H ├┤M├──────────────────────────»
     └────────────┘ ░ ┌───┐     ┌─┴─┐└┬─┬┘└╥┘                          »
q_1: ───────────────░─┤ H ├──■──┤ X ├─┤M├──╫───────────────────────────»
                    ░ └───┘┌─┴─┐└───┘ └╥┘  ║   ┌──────  ┌───┐ ───────┐ »
q_2: ───────────────░──────┤ X ├───────╫───╫───┤ If-0  ─┤ Z ├  End-0 ├─»
                    ░      └───┘       ║   ║   └──╥───  └───┘ ───────┘ »
                                       ║   ║ ┌────╨────┐               »
c: 3/══════════════════════════════════╩═══╩═╡ c_0=0x1 ╞═══════════════»
                                       1   0 └─────────┘               »
«                                ░                  
«q_0: ───────────────────────────░──────────────────
«                                ░                  
«q_1: ───────────────────────────░──────────────────
«       ┌──────  ┌───┐ ───────┐  ░ ┌────────────┐┌─┐
«q_2: ──┤ If-0  ─┤ X ├  End-0 ├──░─┤ post_probe ├┤M├
«       └──╥───  └───┘ ───────┘  ░ └────────────┘└╥┘
«     ┌────╨────┐                                 ║ 
«c: 3/╡ c_1=0x1 ╞═════════════════════════════════╩═
«     └─────────┘                                 2

## Success Probabilities
Over all noise configs and probes, and whether Bob corrects
* If Bob corrects, P(success) is decently high
* If Bob does not, P(success) is basically 50/50
* P(success) decreases as more noise is added: Error(gates), Error(readout) < Error(combined)

In [85]:
for corrected in (True, False):
    print(f"Corrected: {corrected}")
    for noise in NOISE:
        res = {}
        for probe in PROBES:
            qt = quantum_teleportation(probe,  corrected)
            result, tqc = run(qt, noise)
            result_dict = result.get_counts()
            res[probe] = round(marginal_counts(result_dict, [2])["0"] / SHOTS, 4)
        print(f"- {noise}: {res}")
    print(f"{'=' * 10}")

Corrected: True
- ideal: {'0': 1.0, '1': 1.0, '+': 1.0, '-': 1.0, '+i': 1.0, '-i': 1.0}
- gates: {'0': 0.9766, '1': 0.9719, '+': 0.9653, '-': 0.9595, '+i': 0.9595, '-i': 0.9595}
- readout: {'0': 0.9414, '1': 0.9414, '+': 0.9368, '-': 0.9368, '+i': 0.9099, '-i': 0.9099}
- combined: {'0': 0.9216, '1': 0.9172, '+': 0.9087, '-': 0.905, '+i': 0.8804, '-i': 0.8804}
Corrected: False
- ideal: {'0': 0.501, '1': 0.501, '+': 0.501, '-': 0.501, '+i': 0.501, '-i': 0.501}
- gates: {'0': 0.501, '1': 0.501, '+': 0.501, '-': 0.501, '+i': 0.501, '-i': 0.501}
- readout: {'0': 0.499, '1': 0.499, '+': 0.499, '-': 0.499, '+i': 0.499, '-i': 0.499}
- combined: {'0': 0.499, '1': 0.499, '+': 0.499, '-': 0.499, '+i': 0.499, '-i': 0.499}


### Questions
* Why the phase-sensitive probes are useful: To proof the generalisability of the teleportation to multiple bases (X, Y and Z), the probes must be phase-sensitive to show that the teleportation (in the noiseless case) is not only sound in terms of bits, but in correctly transmitting phase as well.

* Why Bob needs Alice’s outcomes: Alice applies a Hadamard measurement on her two qubits after introducing some entanglement with Bob's qubit, causing Bob's qubit to deviate from the original unknown qubit, meaning Bob has to correct this perturbation in order to accurately "receive" the qubit from the teleportation. Thus, Alice must supply Bob with her measurement.

* Why readout errors can affect corrections: Because Bob relies on Alice's readouts to correct, if the readout induces an error, it can cause Bob to correct his qubit wrongly. This may introduce a mismatch between the final state of Bob's qubit and the state of the original unknown qubit, meaning the teleportation fails.

# Problem B: Superdense Coding

In [71]:
TWO_QUBITS = ["00", "01", "10", "11"]

# get entangle gates 
def entangle():
    entangle = QuantumCircuit(2, name="entangle")
    entangle.h(0)
    entangle.cx(0, 1)
    return entangle

def superdense_coding(message):
    # q[0, 1] = e-bits, c[0, 1] = clbits
    q = QuantumCircuit(2, 2)
    q.append(entangle(), [0, 1])
    q.barrier()

    # alice performs ops depending on the message
    if message[0] == "1":
        q.z(0)
    if message[1] == "1":
        q.x(0)

    # alice sends to bob
    q.barrier()

    # bob performs bell measurement
    q.cx(0, 1)
    q.h(0)
    q.measure(0, 1) # flip the order since q0 determines phase flip
    q.measure(1, 0)
    return q

## Circuit
Assuming the message sent is "11" and entangled bit preparation is included

In [73]:
superdense_coding("11").draw()

┌───────────┐ ░ ┌───┐┌───┐ ░      ┌───┐┌─┐
q_0: ┤0          ├─░─┤ Z ├┤ X ├─░───■──┤ H ├┤M├
     │  entangle │ ░ └───┘└───┘ ░ ┌─┴─┐└┬─┬┘└╥┘
q_1: ┤1          ├─░────────────░─┤ X ├─┤M├──╫─
     └───────────┘ ░            ░ └───┘ └╥┘  ║ 
c: 2/════════════════════════════════════╩═══╩═
                                         0   1

## Success Probabilities
Across all noises and messages
* Same phenomena as Part A

In [86]:
for noise in NOISE:
    res = {}
    for message in TWO_QUBITS:
        qsc = superdense_coding(message)
        result, tqc = run(qsc, noise)
        res[message] = result.get_counts()[message] / SHOTS
    print(f"- {noise}: {res}")

- ideal: {'00': 1.0, '01': 1.0, '10': 1.0, '11': 1.0}
- gates: {'00': 0.963134765625, '01': 0.956298828125, '10': 0.958984375, '11': 0.955078125}
- readout: {'00': 0.94189453125, '01': 0.939697265625, '10': 0.94580078125, '11': 0.944091796875}
- combined: {'00': 0.906005859375, '01': 0.89794921875, '10': 0.9072265625, '11': 0.90380859375}


## Repeating `readout`

In [92]:
_message = "10"
_noise = "readout"
_shots = [256, 8192]
_seeds = [7, 8, 9]

for shot in _shots:
    for seed in _seeds:
        qsc = superdense_coding(_message)
        result, tqc = run(qsc, _noise, shot, seed)
        p = result.get_counts()[_message] / shot
        std = np.sqrt(p * (1 - p) / shot)
        print(f"{shot} shots, seed = {seed}: {p} +- {std}")

256 shots, seed = 7: 0.9296875 +- 0.01597955876167341
256 shots, seed = 8: 0.91015625 +- 0.017872361101749213
256 shots, seed = 9: 0.94140625 +- 0.014678923355036537
8192 shots, seed = 7: 0.9427490234375 +- 0.002566814732644567
8192 shots, seed = 8: 0.9420166015625 +- 0.0025821777791048012
8192 shots, seed = 9: 0.9437255859375 +- 0.0025461464732675536
